# Figures

**Channel-Level Diagnostic for Symmetry Breaking in Noisy Equivariant Quantum Neural Networks
by H. Ugail and N. Howard**

This notebook builds the **six** figures
It reads the CSVs produced by the experiment notebook and writes production-quality PNGs.

| File          | Manuscript role                          | Source                                  |
|---------------|------------------------------------------|-----------------------------------------|
| `Figure1.png` | Method workflow schematic                | none (pure illustration)                |
| `Figure2.png` | $U(1)$ phase-symmetry sweep (A/B)        | `sweep_u1.csv`                          |
| `Figure3.png` | $SU(2)$ rotation-symmetry sweep (A/B)    | `sweep_su2.csv`                         |
| `Figure4.png` | Mixed-noise sweep                        | `mixed_sweep.csv`                       |
| `Figure5.png` | $\gamma$-sensitivity ($SU(2)$)            | `sweep_su2.csv` (recomputed at multiple $\gamma$) |
| `Figure6.png` | $SU(2)$ Haar sampling stability          | `robustness_R6_sampling_stability.csv`  |

**Output format.** PNG only at **300 dpi** with `bbox_inches='tight'`. No PDFs.

**Sizing.** Sized for IEEEtran journal columns: 7.16 in (double-column,
two-panel figures) or 3.5 in (single-column).

**Conventions used in axis labels and legends.** Sentence-case capitalisation
on the first word of every axis label; mathematical symbols rendered in math
mode ($\varepsilon$, $\Delta_G$, $C_G$, $q_0$, $|S|$); terminology aligned
with the manuscript captions (the swept variable is called *noise strength*
$\varepsilon$, breaking sites are written $q_0$ rather than `q0`).


## 1. Imports and global style


In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

from google.colab import drive
drive.mount('/content/drive')

CSV_DIR = Path('/content/drive/.../results')
FIG_DIR = Path('/content/drive/MyDrive/.../figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

# IEEE Access two-column journal style. Single-column figure width is
# approximately 3.5 in; double-column span is approximately 7.16 in.
plt.rcParams.update({
    'font.family':       'sans-serif',
    'font.sans-serif':   ['DejaVu Sans', 'Arial', 'Helvetica'],
    'font.size':          9,
    'axes.labelsize':     9.5,
    'axes.titlesize':     10,
    'legend.fontsize':    7,        # was 8 — smaller, more professional
    'legend.framealpha':  0.95,     # higher contrast over data
    'legend.borderpad':   0.3,      # tighter internal padding
    'legend.labelspacing':0.25,
    'legend.handlelength':1.5,
    'legend.handletextpad':0.4,
    'legend.borderaxespad':0.4,
    'xtick.labelsize':    8,
    'ytick.labelsize':    8,
    'axes.linewidth':     0.8,
    'lines.linewidth':    1.4,
    'lines.markersize':   4.5,
    'savefig.bbox':      'tight',
    'savefig.dpi':        300,
    'figure.dpi':         110,
    'pdf.fonttype':       42,
    'ps.fonttype':        42,
})

DOUBLE_COL = 7.16   # inches, IEEE double-column span
SINGLE_COL = 3.5    # inches, IEEE single-column

GAMMA = 5.0
GAMMA_GRID = [2.0, 5.0, 10.0]


def save_figure(fig, name):
    """Write a figure as a 300-dpi PNG named <name>.png in FIG_DIR."""
    path = FIG_DIR / f'{name}.png'
    fig.savefig(path, dpi=300)
    print(f'wrote {path}')


def add_panel_label(ax, label, dx=-0.13, dy=1.06):
    ax.text(dx, dy, label, transform=ax.transAxes,
            fontsize=11, fontweight='bold', va='top', ha='left')


print('Style configured.')
print(f'CSV input:  {CSV_DIR}')
print(f'PNG output: {FIG_DIR}')


## 2. Figure 1 — method workflow schematic

Pure illustration. No data input. Five colour-grouped boxes show the workflow:
ideal equivariant ansatz $\mathcal{V}_\theta$, composition with hardware noise
$\mathcal{E} = \mathcal{N}_\varepsilon \circ \mathcal{V}_\theta$, channel-level
commutator $[\Phi_\mathcal{E}, \Phi_g]$, normalised commutator defect
$\Delta_G \in [0, 1]$, and bounded compliance score $C_G = e^{-\gamma \Delta_G}$.


In [ ]:
fig, ax = plt.subplots(figsize=(DOUBLE_COL, 2.2))
ax.set_xlim(0, 100)
ax.set_ylim(0, 30)
ax.axis('off')

stages = [
    {'x':  2, 'w': 16, 'title': 'Ideal\nequivariant\nansatz',
     'sym': r'$\mathcal{V}_\theta$',                                            'fill': '#e8f1fb'},
    {'x': 22, 'w': 16, 'title': 'Composed\nnoisy\nchannel',
     'sym': r'$\mathcal{E} = \mathcal{N}_\varepsilon\circ\mathcal{V}_\theta$', 'fill': '#e8f1fb'},
    {'x': 42, 'w': 16, 'title': 'Channel-level\ncommutator\nwith ' + r'$\mathcal{U}_g$',
     'sym': r'$[\Phi_\mathcal{E},\Phi_g]$',                                    'fill': '#fbeee8'},
    {'x': 62, 'w': 16, 'title': 'Normalised\ncommutator\ndefect',
     'sym': r'$\Delta_G\in[0,1]$',                                              'fill': '#fbeee8'},
    {'x': 82, 'w': 16, 'title': 'Bounded\ncompliance\nscore',
     'sym': r'$C_G = e^{-\gamma\Delta_G}$',                                     'fill': '#e8f9ec'},
]

BOX_BOTTOM, BOX_TOP = 4, 26
BOX_HEIGHT = BOX_TOP - BOX_BOTTOM

for s in stages:
    box = FancyBboxPatch(
        (s['x'], BOX_BOTTOM), s['w'], BOX_HEIGHT,
        boxstyle='round,pad=0.4,rounding_size=1.5',
        linewidth=1.0, edgecolor='#4a4a4a', facecolor=s['fill'],
    )
    ax.add_patch(box)
    cx = s['x'] + s['w'] / 2
    ax.text(cx, BOX_BOTTOM + BOX_HEIGHT * 0.78, s['title'],
            ha='center', va='center', fontsize=8.5, color='#333333')
    ax.text(cx, BOX_BOTTOM + BOX_HEIGHT * 0.30, s['sym'],
            ha='center', va='center', fontsize=10, color='#222222')

for s_left, s_right in zip(stages[:-1], stages[1:]):
    x_start = s_left['x'] + s_left['w']
    x_end   = s_right['x']
    arrow = FancyArrowPatch(
        (x_start + 0.3, BOX_BOTTOM + BOX_HEIGHT / 2),
        (x_end   - 0.3, BOX_BOTTOM + BOX_HEIGHT / 2),
        arrowstyle='-|>', mutation_scale=12,
        linewidth=1.0, color='#4a4a4a',
    )
    ax.add_patch(arrow)

save_figure(fig, 'Figure1')
plt.show()


## 3. Figure 2 — $U(1)$ phase-symmetry sweep

Two-panel figure spanning both columns. Panel A shows the raw commutator
defect $\Delta_G$, panel B the bounded compliance score
$C_G = \exp(-\gamma \Delta_G)$ at $\gamma = 5$. Depolarising noise and
amplitude damping both commute with phase rotations $e^{i\theta n}$ at the
channel level and stay flat at $\Delta_G = 0$ ($C_G = 1$); only the
coherent $X$ over-rotation on $q_0$ breaks $U(1)$.


In [ ]:
df = pd.read_csv(CSV_DIR / 'sweep_u1.csv')

fig, (axA, axB) = plt.subplots(1, 2, figsize=(DOUBLE_COL, 2.8))

# Panel A: raw defect.
axA.plot(df['eps'], df['delta_depolarising'],  'o-', label='depolarising (symmetric)')
axA.plot(df['eps'], df['delta_amp_damp'],      's-', label='amplitude damping')
axA.plot(df['eps'], df['delta_coherent_X_q0'], '^-', label=r'coherent $X$ on $q_0$')
axA.set_xlabel(r'Noise strength $\varepsilon$')
axA.set_ylabel(r'Commutator defect $\Delta_G$')
# Lift the y-ceiling slightly so the upper-left legend has clear headroom
# above the rising coherent X curve.
ymax_A = max(0.05, 1.25 * float(df[['delta_depolarising','delta_amp_damp','delta_coherent_X_q0']].values.max()))
axA.set_ylim(top=ymax_A)
axA.legend(loc='upper left')
axA.grid(alpha=0.3)
add_panel_label(axA, 'A')

# Panel B: bounded compliance score. Curves end low so 'lower left' clashes
# with the descending coherent X curve. Place legend at the lower right
# instead, in the clear region after the curves have flattened, and lower
# the y-floor a touch so the legend has clean separation.
axB.plot(df['eps'], df['C_G_depolarising'],    'o-', label='depolarising (symmetric)')
axB.plot(df['eps'], df['C_G_amp_damp'],        's-', label='amplitude damping')
axB.plot(df['eps'], df['C_G_coherent_X_q0'],   '^-', label=r'coherent $X$ on $q_0$')
axB.set_xlabel(r'Noise strength $\varepsilon$')
axB.set_ylabel(r'Compliance score $C_G$')
axB.set_ylim([-0.02, 1.05])
axB.axhline(1.0, color='gray', linestyle=':', linewidth=0.8)
axB.legend(loc='lower left')
axB.grid(alpha=0.3)
add_panel_label(axB, 'B')

plt.tight_layout()
save_figure(fig, 'Figure2')
plt.show()


## 4. Figure 3 — $SU(2)$ rotation-symmetry sweep

Same two-panel layout as Fig. 2. The conceptually important contrast with
Fig. 2 is that amplitude damping, which was $U(1)$-equivariant in Fig. 2,
now breaks $SU(2)$ because it is not unitarily covariant. Depolarising
remains $SU(2)$-equivariant (flat at $\Delta_G = 0$); amplitude damping
and coherent $X$ on $q_0$ both register positive defect.


In [ ]:
df = pd.read_csv(CSV_DIR / 'sweep_su2.csv')

fig, (axA, axB) = plt.subplots(1, 2, figsize=(DOUBLE_COL, 2.8))

# Panel A: short labels (long parenthetical "breaks SU(2)" tags removed —
# the contrast with Fig. 2 is now made in the caption rather than crowded
# into the legend, keeping the legend compact).
axA.plot(df['eps'], df['delta_depolarising'],  'o-', label='depolarising (symmetric)')
axA.plot(df['eps'], df['delta_amp_damp'],      's-', label='amplitude damping')
axA.plot(df['eps'], df['delta_coherent_X_q0'], '^-', label=r'coherent $X$ on $q_0$')
axA.set_xlabel(r'Noise strength $\varepsilon$')
axA.set_ylabel(r'Commutator defect $\Delta_G$')
ymax_A = max(0.05, 1.25 * float(df[['delta_depolarising','delta_amp_damp','delta_coherent_X_q0']].values.max()))
axA.set_ylim(top=ymax_A)
axA.legend(loc='upper left')
axA.grid(alpha=0.3)
add_panel_label(axA, 'A')

axB.plot(df['eps'], df['C_G_depolarising'],    'o-', label='depolarising (symmetric)')
axB.plot(df['eps'], df['C_G_amp_damp'],        's-', label='amplitude damping')
axB.plot(df['eps'], df['C_G_coherent_X_q0'],   '^-', label=r'coherent $X$ on $q_0$')
axB.set_xlabel(r'Noise strength $\varepsilon$')
axB.set_ylabel(r'Compliance score $C_G$')
axB.set_ylim([-0.02, 1.05])
axB.axhline(1.0, color='gray', linestyle=':', linewidth=0.8)
axB.legend(loc='lower left')
axB.grid(alpha=0.3)
add_panel_label(axB, 'B')

plt.tight_layout()
save_figure(fig, 'Figure3')
plt.show()


## 5. Figure 4 — mixed-noise sweep

Single-column figure showing $C_G$ for both $U(1)$ and $SU(2)$ as a function
of the coherent $X$ rotation strength on top of a fixed $p = 0.1$
depolarising background. The depolarising background passes through cleanly
(both groups are still equivariant under it); the coherent $X$ ramp drives
the compliance score down for both groups.


In [ ]:
df = pd.read_csv(CSV_DIR / 'mixed_sweep.csv')

fig, ax = plt.subplots(figsize=(SINGLE_COL, 2.6))
ax.plot(df['eps'], df['C_G_u1'],  'o-', label=r'$U(1)$, depol $p=0.1$ + coh. $X$')
ax.plot(df['eps'], df['C_G_su2'], 's-', label=r'$SU(2)$, depol $p=0.1$ + coh. $X$')
ax.set_xlabel(r'Coherent rotation strength $\varepsilon$')
ax.set_ylabel(r'Compliance score $C_G$')
ax.set_ylim([-0.02, 1.05])
ax.axhline(1.0, color='gray', linestyle=':', linewidth=0.8)
ax.legend(loc='lower left')
ax.grid(alpha=0.3)

plt.tight_layout()
save_figure(fig, 'Figure4')
plt.show()


## 6. Figure 5 — $\gamma$-sensitivity for $SU(2)$

Single-column figure recomputing $C_G = \exp(-\gamma \Delta_G)$ at three
values of the compliance temperature $\gamma \in \{2, 5, 10\}$ from the
existing $SU(2)$ defect data. The qualitative ranking of channels —
coherent $X$ on $q_0$ produces the largest compliance drop, amplitude
damping a smaller one — is preserved at every $\gamma$ value, demonstrating
that the diagnostic is not artifactually dependent on a single choice of
compliance temperature. Legend uses a three-column layout (one column per
$\gamma$) so it can fit comfortably below the curves.


In [ ]:
df = pd.read_csv(CSV_DIR / 'sweep_su2.csv')

# Slightly taller figure to accommodate the above-axes legend.
fig, ax = plt.subplots(figsize=(SINGLE_COL, 2.9))
colours = {2.0: 'tab:blue', 5.0: 'tab:green', 10.0: 'tab:purple'}

# Plot in gamma-paired order. The legend sits ABOVE the axes with
# ncol=3, which (with column-major fill) reads as:
#   col 1: coh X gamma=2 / amp damp gamma=2
#   col 2: coh X gamma=5 / amp damp gamma=5
#   col 3: coh X gamma=10 / amp damp gamma=10
for g in GAMMA_GRID:
    cgX = np.exp(-g * df['delta_coherent_X_q0'].to_numpy())
    cgA = np.exp(-g * df['delta_amp_damp'].to_numpy())
    ax.plot(df['eps'], cgX, '^-',  color=colours[g],
            label=fr'coh. $X$, $\gamma{{=}}{int(g)}$')
    ax.plot(df['eps'], cgA, 's--', color=colours[g], alpha=0.75,
            label=fr'amp. damp., $\gamma{{=}}{int(g)}$')

ax.set_xlabel(r'Noise strength $\varepsilon$')
ax.set_ylabel(r'Compliance score $C_G$')
ax.set_ylim(-0.02, 1.05)
ax.axhline(1.0, color='gray', linestyle=':', linewidth=0.7)
ax.grid(alpha=0.3)

# Place legend above the axes. loc='lower center' anchors the
# bottom-centre of the legend at the bbox_to_anchor point, so
# y=1.02 puts the legend just above the axes top and growing upward.
# subplots_adjust(top=...) reserves matching figure-space.
ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1.02),
          ncol=3, fontsize=6.5, columnspacing=0.9,
          handlelength=1.4, borderpad=0.35, labelspacing=0.3)

fig.subplots_adjust(left=0.18, right=0.97, top=0.78, bottom=0.16)
save_figure(fig, 'Figure5')
plt.show()


## 7. Figure 6 — $SU(2)$ Haar sampling stability

Single-column figure built from `robustness_R6_sampling_stability.csv`.
For each test channel and each Haar sample size $|S| \in \{10, 25, 50, 100\}$,
the CSV records the mean and standard deviation of $\Delta_G$ across six
independent draws of the Haar sample set. Equivariant controls (identity,
depolarising) sit flat at $\Delta_G \approx 0$; breaking channels
(amplitude damping, coherent $X$ on $q_0$) sit at clearly non-zero values
with error bars that shrink as $|S|$ grows.


In [ ]:
df = pd.read_csv(CSV_DIR / 'robustness_R6_sampling_stability.csv')

print('channels:    ', sorted(df['channel'].unique()))
print('sample sizes:', sorted(df['n_samples'].unique()))
print()

# Order channels for a readable legend: identity first (zero), then the
# symmetric depolarising control, then the two breakers.
channel_order = [
    'identity',
    'depol_p=0.2',
    'amp_damp_eps=0.2',
    'coherent_X_eps=0.2',
]
channel_label = {
    'identity':           'identity (equivariant)',
    'depol_p=0.2':        r'depolarising $p{=}0.2$ (equivariant)',
    'amp_damp_eps=0.2':   r'amplitude damping, $\varepsilon{=}0.2$',
    'coherent_X_eps=0.2': r'coherent $X$ on $q_0$, $\varepsilon{=}0.2$',
}
channel_marker = {
    'identity':           'o',
    'depol_p=0.2':        'D',
    'amp_damp_eps=0.2':   's',
    'coherent_X_eps=0.2': '^',
}

fig, ax = plt.subplots(figsize=(SINGLE_COL, 2.7))

for ch in channel_order:
    if ch not in df['channel'].values:
        continue
    d = df[df['channel'] == ch].sort_values('n_samples')
    ax.errorbar(
        d['n_samples'], d['mean_Delta'], yerr=d['std_Delta'],
        marker=channel_marker[ch], capsize=2.5, linewidth=1.2,
        markersize=4.5, label=channel_label[ch],
    )

ax.set_xscale('log')
ax.set_xticks([10, 25, 50, 100])
ax.set_xticklabels(['10', '25', '50', '100'])
ax.set_xlabel(r'Haar sample count $|S|$')
ax.set_ylabel(r'Mean $\Delta_G$ ($\pm 1\sigma$)')
ax.set_ylim(-0.005, 0.135)
ax.legend(loc='upper left', fontsize=6)
ax.grid(alpha=0.3)

plt.tight_layout()
save_figure(fig, 'Figure6')
plt.show()


## 8. Done

All six PNG figures have been written to `FIG_DIR`. The cell below lists
them with file sizes for verification.


In [ ]:
for f in sorted(FIG_DIR.glob('Figure*.png')):
    print(f'{f.name:14s}  {f.stat().st_size:>10,} bytes')
